In [ ]:
from pathlib import Path

import torch
from rtnls_enface.utils.plotting import plot_gridfns
from rtnls_models import KeypointsEnsemble  # noqa: F401
from rtnls_models.inference.segmentation import SegmentationEnsemble  # noqa: F401
from rtnls_models.utils.config import load_config  # noqa: F401

## Segmentation of preprocessed images

Here we segment images preprocessed using 0_preprocess.ipynb

In [ ]:
ds_path = Path("../../../samples/fundus")

# input folder. point to a folder with png and/or dicom images
rgb_path = ds_path / "preprocessed_rgb"
ce_path = ds_path / "preprocessed_ce"

# these are the output folders:
av_path = ds_path / "av"
overlays_path = ds_path / "overlays"
discs_path = ds_path / "discs"

device = torch.device("cuda:0")

### Artery-vein segmentation

In [ ]:
av_ensemble = SegmentationEnsemble(load_config("artery_vein"), device=device)
# run the model, pass the path for the outputs
av_ensemble.predict_preprocessed(rgb_path, ce_path, dest_path=av_path, num_workers=2)

### Disc segmentation

In [ ]:
disc_ensemble = SegmentationEnsemble(load_config("disc"), device=device)
disc_ensemble.predict_preprocessed(
    rgb_path, ce_path, dest_path=discs_path, num_workers=2
)

### Fovea detection

In [ ]:
# fovea_model = KeypointsEnsemble(device=device, model_name="r101_b16_001")
# df = fovea_model.predict(npy_path, preprocess=False)
# # the fovea model returns a dataframe with the fovea locations predictions by 5 models (x1,y1),(x2,y2),(x3,y3),(x4,y4),(x5,y5)
# # here we average the predictions of the 5 models before storing the dataframe
# df["mean_x"] = np.mean([df[f"x{i}"] for i in range(0, 5)], axis=0)
# df["mean_y"] = np.mean([df[f"y{i}"] for i in range(0, 5)], axis=0)
# df.to_csv(ds_path / "fovea.csv")

### Plotting the retinas

This will only work if you ran all the models and stored the outputs using the same folder/file names as above

In [ ]:
from vascx.utils.loader import RetinaLoader

loader = RetinaLoader.from_folder(ds_path, fundus_subfolder="preprocessed_rgb")

In [ ]:
plot_gridfns([ret.plot for ret in loader[:8]])